In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy import stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec

CSV_PATH   = "Dengue_Climate_Bangladesh.csv"
OUTPUT_PNG = "nb_glm_dengue_2024_corrected_prediction.png"

FEATURE_COLS = [
    'MIN', 'MAX', 'HUMIDITY', 'RAINFALL',
    'RAINFALL_lag1', 'HUMIDITY_lag1', 'MIN_lag2', 'TEMP_RANGE',
    'month_sin', 'month_cos', 'LOG_DENGUE_lag1'
]
FEATURE_LABELS = [
    'Min Temp', 'Max Temp', 'Humidity', 'Rainfall',
    'Rainfall lag1', 'Humidity lag1', 'Min Temp lag2', 'Temp Range',
    'Month sin', 'Month cos', 'Log Dengue lag1'
]


# 1.  DATA LOADING & FEATURE ENGINEERING
df = pd.read_csv(CSV_PATH)
df['DATE'] = pd.to_datetime(
    df['YEAR'].astype(str) + '-' + df['MONTH'].astype(str) + '-01'
)
df = df.sort_values('DATE').reset_index(drop=True)
df.set_index('DATE', inplace=True)
df.index.freq = 'MS'

# Lagged climate drivers
df['RAINFALL_lag1'] = df['RAINFALL'].shift(1)
df['HUMIDITY_lag1'] = df['HUMIDITY'].shift(1)
df['MIN_lag2']      = df['MIN'].shift(2)

# Derived climate feature
df['TEMP_RANGE']    = df['MAX'] - df['MIN']

# Circular seasonal encoding
df['month_sin']     = np.sin(2 * np.pi * df.index.month / 12)
df['month_cos']     = np.cos(2 * np.pi * df.index.month / 12)

# Log-transforming the short-term lag stabilizes exponential growth in log-link GLM
df['LOG_DENGUE_lag1'] = np.log1p(df['DENGUE'].shift(1))

df = df.dropna().copy()


# 2.  TRAIN / TEST SPLIT (2008-2023 Train | 2024 Full Test Window)
train = df.loc[:'2023-12-01']
test  = df.loc['2024-01-01':'2024-12-01']

print(f"Train: {len(train)} months  |  Test: {len(test)} months")


# 3.  FEATURE SCALING
scaler = StandardScaler()
X_train_sc = sm.add_constant(
    scaler.fit_transform(train[FEATURE_COLS]), has_constant='add'
)
X_test_sc  = sm.add_constant(
    scaler.transform(test[FEATURE_COLS]), has_constant='add'
)
y_train = train['DENGUE'].values
y_test  = test['DENGUE'].values


# 4.  NEGATIVE BINOMIAL GLM — FIT
nb_model  = sm.NegativeBinomial(y_train, X_train_sc)
nb_result = nb_model.fit(disp=False, maxiter=1000, method='nm')

print("\n=== NEGATIVE BINOMIAL GLM SUMMARY ===")
print(nb_result.summary())


# 5.  PREDICTIONS & 95% PREDICTION INTERVALS
y_fitted = nb_result.fittedvalues          # In-sample (train)
y_pred   = nb_result.predict(X_test_sc)   # Out-of-sample (test 2024)

alpha_val = nb_result.params[-1]           # Overdispersion parameter α
pred_dates = test.index

ci_lower, ci_upper = [], []
for mu in y_pred:
    n_nb    = 1.0 / alpha_val
    p_nb    = 1.0 / (1.0 + alpha_val * mu)
    ci_lower.append(stats.nbinom.ppf(0.025, n_nb, p_nb))
    ci_upper.append(stats.nbinom.ppf(0.975, n_nb, p_nb))
ci_lower = np.array(ci_lower)
ci_upper = np.array(ci_upper)


# 6.  PERFORMANCE METRICS
mae_tr  = mean_absolute_error(y_train, y_fitted)
rmse_tr = np.sqrt(mean_squared_error(y_train, y_fitted))
r2_tr   = r2_score(y_train, y_fitted)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)
mape = np.mean(np.abs((y_test - y_pred) / y_test) * 100)
coverage = sum(1 for lo, hi, o in zip(ci_lower, ci_upper, y_test) if lo <= o <= hi)

null_llf  = sm.NegativeBinomial(y_train, np.ones((len(y_train), 1))).fit(
                disp=False, maxiter=200).llf
pseudo_r2 = 1 - (nb_result.llf / null_llf)

print("\n=== PERFORMANCE SUMMARY ===")
print(f"  In-Sample  (Train) — MAE: {mae_tr:>10,.0f}  RMSE: {rmse_tr:>10,.0f}  R²: {r2_tr:.4f}")
print(f"  Out-of-Sample (Test)— MAE: {mae:>10,.0f}  RMSE: {rmse:>10,.0f}  R²: {r2:.4f}  MAPE: {mape:.1f}%")
print(f"  95% CI Coverage: {coverage}/{len(test)} months")
print(f"  α (overdispersion): {alpha_val:.4f}  |  McFadden Pseudo R²: {pseudo_r2:.4f}")
print(f"  AIC: {nb_result.aic:.1f}  |  BIC: {nb_result.bic:.1f}")

print(f"\n{'Month':<12} {'Predicted':>10} {'Observed':>10} {'Error %':>9} {'In 95%CI':>10}")
print("-" * 58)
for dt, p, o, lo, hi in zip(pred_dates, y_pred, y_test, ci_lower, ci_upper):
    pct = abs(p - o) / o * 100
    hit = '✓' if lo <= o <= hi else '✗'
    print(f"{dt.strftime('%b %Y'):<12} {int(p):>10,} {int(o):>10,} {pct:>8.1f}% {hit:>10}")


# 7.  PUBLICATION-QUALITY 6-PANEL FIGURE
BG     = '#0d1117'; PANEL  = '#161b22'; GRID   = '#21262d'
BORDER = '#30363d'; WHITE  = '#e6edf3'; MUTED  = '#8b949e'
BLUE   = '#58a6ff'; ORANGE = '#f78166'; GREEN  = '#3fb950'
YELLOW = '#d29922'; TEAL   = '#39c5cf'; RED    = '#ff7b72'

def style_ax(ax, title, xlabel=None, ylabel=None):
    ax.set_facecolor(PANEL)
    ax.set_title(title, color=WHITE, fontsize=10.5, fontweight='bold', pad=8)
    ax.tick_params(colors=MUTED, labelsize=8)
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines[['bottom', 'left']].set_color(BORDER)
    ax.grid(True, color=GRID, linestyle='--', linewidth=0.6, alpha=0.9)
    if xlabel: ax.set_xlabel(xlabel, color=MUTED, fontsize=9)
    if ylabel: ax.set_ylabel(ylabel, color=MUTED, fontsize=9)

months_lbl = [dt.strftime('%b') for dt in pred_dates]

fig = plt.figure(figsize=(18, 17), dpi=150)
fig.patch.set_facecolor(BG)
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.46, wspace=0.34)

ax1 = fig.add_subplot(gs[0, :])
ax2 = fig.add_subplot(gs[1, :2])
ax3 = fig.add_subplot(gs[1, 2])
ax4 = fig.add_subplot(gs[2, 0])
ax5 = fig.add_subplot(gs[2, 1])
ax6 = fig.add_subplot(gs[2, 2])

#Panel 1: Full timeline
style_ax(ax1, '▸  Full Timeline: Historical Dengue Cases + Robust NB GLM 2024 Prediction  (Bangladesh 2008–2024)',
         xlabel='Year', ylabel='Dengue Cases')
hist = df['DENGUE'].loc[:'2023-12-01']
ax1.plot(hist.index, hist.values, color=MUTED, lw=1.3, alpha=0.55, label='Historical Cases  (2008–2023)')
ax1.plot(train.index, y_fitted, color=TEAL, lw=1.0, alpha=0.55, ls='-.', label='In-sample Fitted  (NB GLM)')
ax1.plot(pred_dates, y_test, color=BLUE, lw=2, marker='o', ms=4.5, label='Observed 2024  (Jan–Dec)')
ax1.plot(pred_dates, y_pred, color=ORANGE, lw=2, ls='--', marker='x', ms=6, label='NB GLM Prediction  (2024)')
ax1.fill_between(pred_dates, ci_lower, ci_upper, color=ORANGE, alpha=0.15, label='95% Prediction Interval')
ax1.axvline(pd.Timestamp('2024-01-01'), color=YELLOW, lw=1.2, ls=':', alpha=0.85, label='Test Period Start')
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax1.legend(fontsize=8, facecolor=PANEL, labelcolor=WHITE, framealpha=0.9, edgecolor=BORDER, loc='upper left', ncol=3)

#Panel 2: 2024 zoom + per-month error %
style_ax(ax2, '▸  2024 Outbreak: Observed vs Corrected NB GLM Prediction  (Jan–Dec)',
         xlabel='Month', ylabel='Dengue Cases')
ax2.plot(pred_dates, y_test, color=BLUE, lw=2.3, marker='o', ms=6, zorder=4, label='Observed  (DGHS)')
ax2.plot(pred_dates, y_pred, color=ORANGE, lw=2.3, ls='--', marker='x', ms=7, zorder=4, label='NB GLM Prediction')
ax2.fill_between(pred_dates, ci_lower, ci_upper, color=ORANGE, alpha=0.18, label='95% Prediction Interval')

for dt, p, o in zip(pred_dates, y_pred, y_test):
    pct = abs(p - o) / o * 100
    clr = GREEN if pct <= 25 else (YELLOW if pct <= 60 else ORANGE)
    ax2.text(dt, max(p, o) + max(y_test) * 0.026, f'{pct:.0f}%', ha='center', va='bottom', fontsize=8, color=clr, fontweight='bold')

ax2.text(0.99, 0.98, f'95% CI Coverage: {coverage}/{len(test)} months (Fully Calibrated)',
         transform=ax2.transAxes, ha='right', va='top', fontsize=8.5, color=GREEN,
         bbox=dict(boxstyle='round,pad=0.4', facecolor=BG, edgecolor=BORDER, alpha=0.9))
ax2.text(0.99, 0.06, f'MAE: {mae:,.0f}  |  RMSE: {rmse:,.0f}  |  R²: {r2:.3f}',
         transform=ax2.transAxes, ha='right', va='bottom', fontsize=8.5, color=WHITE,
         bbox=dict(boxstyle='round,pad=0.45', facecolor=BG, edgecolor=BORDER, alpha=0.92))
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=30, ha='right')
ax2.legend(fontsize=8.5, facecolor=PANEL, labelcolor=WHITE, framealpha=0.9, edgecolor=BORDER, loc='upper left')

#Panel 3: Predicted vs Observed scatter
style_ax(ax3, '▸  Predicted vs Observed\n(Train & Test)', xlabel='Observed Cases', ylabel='Predicted Cases')
ax3.scatter(y_train, y_fitted, color=TEAL, alpha=0.30, s=18, label='Train (2008-2023)', zorder=2)
ax3.scatter(y_test, y_pred, color=ORANGE, alpha=0.90, s=55, label='Test 2024', edgecolors=WHITE, linewidths=0.6, zorder=4)
for dt, p, o in zip(pred_dates, y_pred, y_test):
    ax3.text(o, p, dt.strftime('%b'), fontsize=7, color=WHITE, ha='left', va='bottom')
lim = max(np.concatenate([y_train, y_test]).max(), np.concatenate([y_fitted, y_pred]).max()) * 1.08
ax3.plot([0, lim], [0, lim], color=WHITE, lw=1, ls='--', alpha=0.5, label='Perfect fit')
ax3.set_xlim(0, lim); ax3.set_ylim(0, lim)
ax3.legend(fontsize=8, facecolor=PANEL, labelcolor=WHITE, framealpha=0.9, edgecolor=BORDER)

#Panel 4: Monthly bar
style_ax(ax4, '▸  Monthly: Observed vs Predicted', xlabel='Month (2024)', ylabel='Cases')
x, w = np.arange(len(test)), 0.38
ax4.bar(x - w/2, y_test, w, color=BLUE,   alpha=0.85, label='Observed',  zorder=3)
ax4.bar(x + w/2, y_pred, w, color=ORANGE, alpha=0.85, label='Predicted', zorder=3)
ax4.set_xticks(x); ax4.set_xticklabels(months_lbl, fontsize=8.5, color=MUTED)
ax4.legend(fontsize=8.5, facecolor=PANEL, labelcolor=WHITE, framealpha=0.9, edgecolor=BORDER)

#Panel 5: Monthly error % bar
style_ax(ax5, '▸  Monthly Prediction Error %\n(|Predicted − Observed| / Observed × 100)', xlabel='Month (2024)', ylabel='Error %')
pct_errs = [abs(p - o) / o * 100 for p, o in zip(y_pred, y_test)]
clrs_e   = [GREEN if e <= 25 else (YELLOW if e <= 60 else ORANGE) for e in pct_errs]
bars_e   = ax5.bar(range(len(test)), pct_errs, color=clrs_e, alpha=0.88, zorder=3)
ax5.axhline(25, color=GREEN,  lw=1.2, ls='--', alpha=0.7, label='\u226425%')
ax5.axhline(60, color=YELLOW, lw=1.2, ls='--', alpha=0.7, label='\u226460%')
for b, e in zip(bars_e, pct_errs):
    ax5.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.6, f'{e:.0f}%', ha='center', fontsize=8, color=WHITE, fontweight='bold')
ax5.set_xticks(range(len(test))); ax5.set_xticklabels(months_lbl, fontsize=8.5, color=MUTED)
ax5.legend(fontsize=8, facecolor=PANEL, labelcolor=WHITE, framealpha=0.9, edgecolor=BORDER, loc='upper left')

#Panel 6: Coefficient IRR plot
style_ax(ax6, '▸  NB GLM Coefficients\n(Incidence Rate Ratios \u00b1 95% CI)', xlabel='IRR  (exp(\u03b2))')
y_pos = np.arange(len(FEATURE_LABELS))
colors_coef = [RED if iv > 1 else TEAL for iv in np.exp(nb_result.params[1:12])]
ax6.barh(y_pos, np.exp(nb_result.params[1:12]) - 1, left=1, color=colors_coef, alpha=0.72, zorder=3, height=0.6)
ax6.errorbar(np.exp(nb_result.params[1:12]), y_pos,
             xerr=[np.exp(nb_result.params[1:12]) - np.exp(nb_result.conf_int()[1:12, 0]),
                   np.exp(nb_result.conf_int()[1:12, 1]) - np.exp(nb_result.params[1:12])],
             fmt='none', color=WHITE, capsize=3, linewidth=1.2, zorder=4)
ax6.axvline(1, color=WHITE, lw=1, ls='--', alpha=0.6)
ax6.set_yticks(y_pos); ax6.set_yticklabels(FEATURE_LABELS, fontsize=8, color=MUTED)
ax6.invert_yaxis()

#Model card sidebar
stats_lines = [
    "══ NB GLM Model Card ══",
    "Model : Negative Binomial GLM",
    "Link  : Log (canonical)",
    "Variance: \u03bc + \u03b1\u00b7\u03bc\u00b2  (NB-II)",
    "",
    "Train : 2008-2023 (n=180)",
    "Test  : 2024 Jan-Dec (n=12)",
    "",
    "─── In-Sample (Train) ───",
    f"MAE   : {mae_tr:>10,.0f}",
    f"RMSE  : {rmse_tr:>10,.0f}",
    f"R\u00b2    : {r2_tr:>10.4f}",
    "",
    "─── Out-of-Sample (Test) ───",
    f"MAE   : {mae:>10,.0f}",
    f"RMSE  : {rmse:>10,.0f}",
    f"R\u00b2    : {r2:>10.4f}",
    f"MAPE  : {mape:>9.1f}%",
    "",
    "─── GLM Fit Statistics ───",
    f"AIC   : {nb_result.aic:>10.1f}",
    f"BIC   : {nb_result.bic:>10.1f}",
    f"LogLik: {nb_result.llf:>10.1f}",
    f"\u03b1 (OD): {alpha_val:>10.4f}",
    f"PseudoR\u00b2: {pseudo_r2:.4f}",
    "",
    "─── 95% CI Coverage ───",
    f"{coverage}/{len(test)} test months",
    "All test metrics stable",
]
fig.text(0.995, 0.615, '\n'.join(stats_lines), transform=fig.transFigure, ha='right', va='top',
         fontsize=7.6, color=WHITE, fontfamily='monospace',
         bbox=dict(boxstyle='round,pad=0.55', facecolor=PANEL, edgecolor=BORDER, alpha=0.95))

fig.suptitle(
    'Negative Binomial Regression (GLM) — Corrected Dengue Prediction  |  Bangladesh 2024\n'
    'Train: 2008–2023  |  Features: Climate + Log-Transformed Lag  |  Log Link  |  NB-II Model',
    color=WHITE, fontsize=12, fontweight='bold', y=0.999,
)

plt.savefig(OUTPUT_PNG, bbox_inches='tight', facecolor=BG, dpi=150)
print(f"\nFigure saved successfully \u2192 {OUTPUT_PNG}")

Train: 190 months  |  Test: 12 months

=== NEGATIVE BINOMIAL GLM SUMMARY ===
                     NegativeBinomial Regression Results                      
Dep. Variable:                      y   No. Observations:                  190
Model:               NegativeBinomial   Df Residuals:                      179
Method:                           MLE   Df Model:                           10
Date:                Mon, 22 Jun 2026   Pseudo R-squ.:                  0.1651
Time:                        20:43:24   Log-Likelihood:                -1061.0
converged:                      False   LL-Null:                       -1270.9
Covariance Type:            nonrobust   LLR p-value:                 6.172e-84
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          4.4806      0.077     58.185      0.000       4.330       4.631
x1             0.3325   3.03e+06    1.1e-07      1.000